# HPO SolarSystemLander

Elise HPO (8D) from `hpo/designs/design.md`.

## Set up

### Set up Colab

In [1]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

from hpo.notebook.colab import setup_colab

COLAB = setup_colab()

### Set up HPO

In [ ]:
# cell: hpo-setup # requires: colab-setup

import torch
from hpo.notebook.colab import prepare_storage

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBSERVATION_MODE = "8d"
STUDY_SERIES_NAME = f"solar_system_lander_{OBSERVATION_MODE}_elise"
STUDY_SERIES_STORAGE = prepare_storage(COLAB, STUDY_SERIES_NAME)

## Optimize HPs iteratively

### Define baseline

Define incumbent HPs

#### Alternative 1: Set manually

In [ ]:
# cell: baseline-manual

from hpo.hyperparams import HP
from hpo.study import Baseline

_BASELINE_PARAMS = {
    HP.LEARNING_RATE: 0.0022727854024196057,
    HP.BATCH_SIZE: 512,
    HP.EPS_END: 0.047716002108220544,
    HP.EPS_DECAY: 43_214,
    HP.GAMMA: 0.99,
    HP.TAU: 0.005,
    HP.LEARNING_STARTS: 2_500,
    HP.OPTIMIZE_EVERY: 2,
    HP.REPLAY_MEMORY_CAPACITY: 400_000,
    HP.NUM_EPISODES: 500,
}

baseline = Baseline(params=_BASELINE_PARAMS, score=118.09342221642783)

#### Alternative 2: Read from DB

In [ ]:
# cell: baseline-db

from hpo.study import Baseline

baseline = Baseline.from_database("runs/previous.db", "s4")

baseline.params
baseline.score

### Create study runner

In [ ]:
# cell: objective-config # requires: colab-setup, hpo-setup

from hpo.checkpointing import ObjectiveHookFactory
from hpo.objective import ObjectiveConfig
from hpo.solar_system_lander.environment import EnvFactory

ENV_FACTORY = EnvFactory(OBSERVATION_MODE)
_EARLY_STOPPING_SCORE = -250.0  # optional; default: -250.0; disable: None
OBJECTIVE_CFG = ObjectiveConfig(
    environment_factory=ENV_FACTORY,
    num_envs=20,
    device=device,
    eval_episodes=10,
    early_stopping_score=_EARLY_STOPPING_SCORE,
    hooks=ObjectiveHookFactory(
        checkpoint_dir=COLAB.local_study_dir / f"{STUDY_SERIES_NAME}_checkpoints",
        best_eval_archive_dir=COLAB.drive_study_dir / "best_checkpoints" / STUDY_SERIES_NAME,
        window=100,
    ),
)

In [ ]:
# cell: study-runner # requires: hpo-setup, baseline, objective-config

from hpo.evaluation.dashboard import Dashboard
from hpo.study import StudyRunner

_TRAINING_SCORE_MIN = -500.0  # optional; default: -500.0; disable: None

runner = StudyRunner(
    database_path=lambda _name: STUDY_SERIES_STORAGE.database_path,
    objective_cfg=OBJECTIVE_CFG,
    baseline=baseline,
    reporter=Dashboard(render_mode="safe", training_score_min=_TRAINING_SCORE_MIN),
    study_attrs=ENV_FACTORY.metadata(),
    robust_candidates=5,
    extra_seeds=(1001, 1002, 1003, 1004),
    sync_fn=STUDY_SERIES_STORAGE.backup,
)

### Run studies

In [ ]:
# cell: run-s1-flight-hours # requires: study-runner

def _suggest_params(trial, _incumbent_params):
    trial.suggest_categorical(HP.NUM_EPISODES, [500, 750, 1_000])

runner.run("s1_flight_hours", _suggest_params, 25)

In [ ]:
# cell: run-s2-exploration # requires: study-runner

def _suggest_params(trial, _incumbent_params):
    trial.suggest_float(HP.EPS_END, 0.03, 0.07)
    trial.suggest_int(HP.EPS_DECAY, 30_000, 150_000, log=True)

runner.run("s2_exploration", _suggest_params, 25)

## ──────────────────────────────

solar_system_lander_9d_elise_3

## ──────────────────────────────

## Select best checkpoint

In [ ]:
# cell: select-best-checkpoint # requires: hpo-setup

import optuna

from hpo.checkpointing import best_checkpoint

def _best_checkpoint_for(study_name):
    _study = optuna.load_study(
        study_name=study_name,
        storage=f"sqlite:///{STUDY_SERIES_STORAGE.database_path}",
    )
    return best_checkpoint(_study)

_best = _best_checkpoint_for("s2_exploration")
_best

## Reload studies

In [ ]:
# cell: reload-studies # requires: hpo-setup

# Run only after the runtime environment has been reset.
import optuna

def _load_study(name):
    return optuna.load_study(study_name=name, storage=f"sqlite:///{STUDY_SERIES_STORAGE.database_path}")

study1 = _load_study("s1_flight_hours")
study2 = _load_study("s2_exploration")

## Analyze results

In [ ]:
# cell: compare-reloaded-studies # requires: reload-studies

import pandas as pd

_studies = [study1, study2]
_labels = ["S1", "S2"]

_rows = []
for _label, _study in zip(_labels, _studies):
    _rows.append({
        "study": _label,
        "score": _study.user_attrs["incumbent_score"],
    })

display(pd.DataFrame(_rows).set_index("study").style.format("{:.1f}"))

### Is the lander still learning at the end?

In [ ]:
# cell: load-study-for-analysis # requires: hpo-setup

from pathlib import Path

import optuna
import pandas as pd
import plotly.express as px

_tmp_dir = Path("hpo/tmp")
if not _tmp_dir.exists():
    _tmp_dir = Path("../tmp")
_database_path = (_tmp_dir / f"{STUDY_SERIES_NAME}.db").resolve()
study = optuna.load_study(
    study_name="s2_exploration",
    storage=f"sqlite:///{_database_path}",
)
_params = study.user_attrs.get("robust_best_params")
_trials = [
    _trial for _trial in study.trials
    if _trial.state.name == "COMPLETE" and _trial.params == _params
]
#_trial = max(_trials, key=lambda _trial: _trial.value) if _trials else study.best_trial
_trial = study.best_trial

_returns = _trial.user_attrs["training_curve"]["episode_returns"]
_window = min(100, len(_returns) // 2)
_curve = pd.DataFrame({"Episode return": _returns})
_curve[f"Mean ({_window} episodes)"] = _curve["Episode return"].rolling(_window).mean()

display(px.line(_curve, labels={"index": "Episode", "value": "Return"}))

_previous_mean = _curve["Episode return"].iloc[-2 * _window:-_window].mean()
_final_mean = _curve["Episode return"].iloc[-_window:].mean()
print(f"Previous {_window} episodes: {_previous_mean:.1f}")
print(f"Final {_window} episodes:    {_final_mean:.1f}")
print(f"Change:                    {_final_mean - _previous_mean:+.1f}")

### Should the trials have trained longer?

In [ ]:
# cell: estimate-alpha

import numpy as np

def estimate_alpha(returns, window):
    returns = np.asarray(returns)
    means = [
        returns[-3 * window:-2 * window].mean(),
        returns[-2 * window:-window].mean(),
        returns[-window:].mean(),
    ]
    previous_gain = means[1] - means[0]
    latest_gain = means[2] - means[1]
    return np.nan if np.isclose(previous_gain, 0) else latest_gain / previous_gain

In [ ]:
# cell: plot-training-extension # requires: load-study-for-analysis, estimate-alpha

import numpy as np
import plotly.graph_objects as go

_rows = []
for _trial in study.trials:
    if _trial.state.name != "COMPLETE" or "training_curve" not in _trial.user_attrs:
        continue

    _returns = np.asarray(_trial.user_attrs["training_curve"]["episode_returns"])
    _window = min(50, len(_returns) // 3)
    _previous = _returns[-2 * _window:-_window]
    _final = _returns[-_window:]
    _improvement = _final.mean() - _previous.mean()
    _error = 1.96 * np.sqrt(
        _previous.var(ddof=1) / _window + _final.var(ddof=1) / _window
    )
    _alpha = estimate_alpha(_returns, _window)
    _rows.append((
        _trial.number,
        _improvement,
        _error,
        _improvement > _error,
        _alpha,
    ))

_figure = go.Figure(go.Bar(
    x=[_row[0] for _row in _rows],
    y=[_row[1] for _row in _rows],
    error_y={"type": "data", "array": [_row[2] for _row in _rows]},
    marker_color=["green" if _row[3] else "gray" for _row in _rows],
    text=[f"α={_row[4]:.1f}" if np.isfinite(_row[4]) else "α=n/a" for _row in _rows],
    textposition="outside",
    textfont={"size": 14},
))
_figure.add_hline(y=0, line_color="black")
_figure.update_layout(
    xaxis_title="Trial",
    yaxis_title="Change in mean episode return",
    title=(
        "Green: evidence that longer training may help"
        "<br><sup>Balken: Differenz der Mittelwerte der letzten zwei Fenster</sup>"
    ),
)
_figure.show()

print("window = " + str(_window))

## Tools

In [ ]:
# cell: inspect-optuna-db

# qq2

# Was ist in einer bestimmten Optuna DB?
# Eingabe: _db_path

from pathlib import Path

if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

import optuna
import pandas as pd


_db_path = "/content/drive/MyDrive/rl_lab/hpo/solar_system_lander_9d_elise_3.db" 

_storage = f"sqlite:///{_db_path}"

_rows = []
for _summary in optuna.study.get_all_study_summaries(storage=_storage):
    _study = optuna.load_study(study_name=_summary.study_name, storage=_storage)
    _rows.append({
        "study": _summary.study_name,
        "robust_best_score": _study.user_attrs.get("robust_best_score"),
        "robust_best_params": _study.user_attrs.get("robust_best_params"),
        "incumbent_score": _study.user_attrs.get("incumbent_score"),
        "incumbent_params": _study.user_attrs.get("incumbent_params"),
    })

pd.DataFrame(_rows)

In [ ]:
# cell: import-db-summary

from hpo.notebook.optuna import db_summary

In [ ]:
# cell: show-db-summary # requires: import-db-summary

_db_path = "/content/drive/MyDrive/rl_lab/hpo/solar_system_lander_9d_elise_3.db"

db_summary(_db_path)